# 한양여대 모두의 AI · NEXT AI 해커톤 · GPU 오픈 모델 실습

위에서부터 순서대로 **Shift + Enter**로 실행합니다. 코드는 모두 이 노트북 안에 있습니다.

- **값 바꾸기:** 셀 맨 위의 `바꿔 볼 값` 부분을 고치고 그 셀만 다시 실행합니다. 결과가 바로 아래에 나옵니다.
- **접힌 셀:** 모델 준비처럼 긴 코드는 접어 두었습니다. 접힌 채로 실행해도 됩니다. 셀 왼쪽의 파란 막대를 누르면 펼쳐집니다.
- **모델 전환:** 다음 모델로 넘어가기 전에 `모델 해제` 셀을 실행합니다.
- **모델 파일:** 서버에 모델이 없으면 준비 셀이 내 홈 폴더로 자동으로 내려받습니다. 처음 한 번은 몇 분 걸릴 수 있습니다.
- **실행 중:** 셀 왼쪽이 `[*]`이면 기다립니다. 같은 셀을 반복 실행하지 않습니다.
- **오류:** 메시지를 남기고 스태프에게 알립니다.

## 0. 환경 확인

**0-1 환경 확인** GPU 이름과 메모리, 모델 폴더를 확인합니다.

In [1]:
import os
import sys
from importlib import metadata

print('Python:', sys.version.split()[0])
for package in ('torch', 'transformers', 'diffusers', 'accelerate', 'huggingface-hub', 'imageio', 'imageio-ffmpeg'):
    try:
        print(package, metadata.version(package))
    except metadata.PackageNotFoundError:
        print(package, '없음')
try:
    import torch
    print('CUDA 사용 가능:', torch.cuda.is_available())
    if torch.cuda.is_available():
        print('GPU:', torch.cuda.get_device_name(0))
        print('GPU 전체 메모리(GB):', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))
        free, total = torch.cuda.mem_get_info()
        print('지금 사용 가능한 GPU 메모리(GB):', round(free / 1024**3, 2), '← 다른 사용자와 공유 중이면 전체보다 작습니다')
except ImportError:
    print('torch를 불러올 수 없어 GPU를 확인하지 못했습니다.')
for name in ['qwen2.5-1.5b-instruct', 'sdxl-turbo', 'musicgen-small', 'stable-video-diffusion-img2vid']:
    found = [p for p in (os.path.join('/opt/models', name), os.path.expanduser(os.path.join('~/models', name))) if os.path.isdir(p)]
    print('모델:', name, '확인됨 ' + found[0] if found else '없음: 준비 셀에서 자동으로 내려받습니다')
try:
    import imageio_ffmpeg
    print('ffmpeg(영상 저장용):', imageio_ffmpeg.get_ffmpeg_exe())
except Exception as err:  # 영상 실습을 하지 않으면 없어도 됩니다.
    print('ffmpeg(영상 저장용): 없음 -', type(err).__name__)

Python: 3.14.10
torch 2.15.0
transformers 없음
diffusers 없음
accelerate 없음
huggingface-hub 없음
imageio 없음
imageio-ffmpeg 없음
CUDA 사용 가능: False
모델: qwen2.5-1.5b-instruct 없음: 준비 셀에서 자동으로 내려받습니다
모델: sdxl-turbo 없음: 준비 셀에서 자동으로 내려받습니다
모델: musicgen-small 없음: 준비 셀에서 자동으로 내려받습니다
모델: stable-video-diffusion-img2vid 없음: 준비 셀에서 자동으로 내려받습니다
ffmpeg(영상 저장용): 없음 - ModuleNotFoundError


## 1. 허깅페이스 탐색

GPU를 쓰지 않습니다.

**1-1 허브 검색** `SEARCH`, `TASK`, `SORT`를 바꿔 다시 실행합니다. 서버가 인터넷에 연결되지 않으면 로컬 모델 목록만 나옵니다.

In [3]:
# 서버가 인터넷에 연결되어 있지 않으면 검색을 건너뛰고 /opt/models 의 로컬 모델 목록을 보여 줍니다.
import os
from pathlib import Path

# 바꿔 볼 값: 검색어·작업 종류·정렬 기준
SEARCH = 'Qwen2.5'            # 모델 이름에 들어갈 단어 (예: 'Llama', 'gemma', 'sdxl')
TASK = 'text-generation'      # 허브의 작업 분류 (text-generation, text-to-image, text-to-audio, image-to-video)
SORT = 'downloads'            # 'downloads' 또는 'likes'
LIMIT = 10

def gb(num_bytes):
    return round(num_bytes / 1024**3, 2)

def search_hub():
    from huggingface_hub import HfApi
    api = HfApi()
    print(f'허브 검색: search={SEARCH!r} task={TASK!r} sort={SORT} limit={LIMIT}')
    models = api.list_models(search=SEARCH, pipeline_tag=TASK, sort=SORT, limit=LIMIT)
    rows = []
    for m in models:
        rows.append((m.id, m.downloads or 0, m.likes or 0))
    print(f"\n{'모델 ID':48} {'다운로드':>12} {'좋아요':>8}")
    for model_id, downloads, likes in rows:
        print(f'{model_id:48} {downloads:>12,} {likes:>8,}')
    return [r[0] for r in rows]

def describe(model_id):
    """모델 카드의 핵심 정보: 라이선스·태그·파일 크기 합계. 다운로드는 하지 않습니다."""
    from huggingface_hub import HfApi
    info = HfApi().model_info(model_id, files_metadata=True)
    total = sum((s.size or 0) for s in info.siblings)
    license_tags = [t for t in info.tags if t.startswith('license:')]
    print(f'\n[{model_id}]')
    print('  파이프라인:', info.pipeline_tag)
    print('  라이선스  :', ', '.join(license_tags) or '태그 없음(모델 카드 본문 확인)')
    print('  가중치 파일 합계(GB):', gb(total), ' ← 디스크와 VRAM 추정의 출발점')
    print('  게이트 모델:', '예 (동의·토큰 필요)' if getattr(info, 'gated', False) else '아니오')
    weights = [s for s in info.siblings if s.rfilename.endswith(('.safetensors', '.bin', '.gguf'))]
    for s in weights[:8]:
        print(f'    {s.rfilename:50} {gb(s.size or 0):>6} GB')
    if len(weights) > 8:
        print(f'    ... 외 {len(weights) - 8}개')

def list_local():
    roots = [r for r in (Path('/opt/models'), Path.home() / 'models') if r.is_dir()]
    print('\n로컬 모델(/opt/models, ~/models):')
    if not roots:
        print('  아직 없습니다. 실습 준비 셀에서 자동으로 내려받습니다.')
        return
    for folder in sorted(p for r in roots for p in r.iterdir() if p.is_dir()):
        size = sum(f.stat().st_size for f in folder.rglob('*') if f.is_file())
        has_cfg = (folder / 'config.json').exists() or (folder / 'model_index.json').exists()
        print(f'  {folder.name:45} {gb(size):>7} GB  {"설정 파일 있음" if has_cfg else "설정 파일 없음"}')

online = False
if os.environ.get('HF_HUB_OFFLINE') == '1':
    print('HF_HUB_OFFLINE=1 입니다. 허브 검색을 건너뜁니다.')
else:
    try:
        import socket
        socket.create_connection(('huggingface.co', 443), timeout=3).close()
        online = True
    except OSError as err:
        print('huggingface.co 에 연결할 수 없습니다:', err)
        print('→ 검색은 각자 노트북 브라우저에서 https://huggingface.co/models 로 진행하세요.')

if online:
    try:
        found = search_hub()
        for model_id in found[:2]:
            describe(model_id)
    except Exception as err:  # 네트워크·권한 오류가 나도 수업을 멈추지 않습니다.
        print('허브 조회 중 오류:', type(err).__name__, err)
list_local()
print('\n오늘 실습은 /opt/models 의 사전 배포 모델만 사용합니다. 위 검색은 탐색 방법을 익히기 위한 것입니다.')

허브 조회 중 오류: ModuleNotFoundError No module named 'huggingface_hub'

로컬 모델(/opt/models, ~/models):
  아직 없습니다. 실습 준비 셀에서 자동으로 내려받습니다.

오늘 실습은 /opt/models 의 사전 배포 모델만 사용합니다. 위 검색은 탐색 방법을 익히기 위한 것입니다.


**1-2 모델 파라미터 확인** 층 수, 파라미터 수, 생성 기본값, 예상 VRAM을 읽습니다.

In [8]:
# 구조(층 수·은닉 크기·어휘 수), 파라미터 수, 생성 기본값, 예상 VRAM을 출력합니다.
import json
import struct
from pathlib import Path

MODEL_NAMES = ['qwen2.5-1.5b-instruct', 'sdxl-turbo', 'musicgen-small', 'stable-video-diffusion-img2vid']
MODEL_DIRS = [str(Path('/opt/models') / n) if (Path('/opt/models') / n).is_dir() else str(Path.home() / 'models' / n) for n in MODEL_NAMES]

BYTES_PER_PARAM = {'FP32': 4, 'FP16/BF16': 2, 'INT8': 1, 'INT4(약)': 0.55}

def read_json(path):
    try:
        return json.loads(Path(path).read_text(encoding='utf-8'))
    except (OSError, ValueError):
        return None

def weight_files(folder):
    """세어야 할 safetensors 파일만 고릅니다.
    - 같은 폴더에 fp16 변형(.fp16.safetensors)과 기본 변형이 함께 있으면 한쪽(fp16)만 셉니다.
    - Diffusers 저장소(model_index.json)의 최상위에 있는 단일 파일 체크포인트는 구성 요소와 중복이므로 제외합니다."""
    folder = Path(folder)
    is_diffusers = (folder / 'model_index.json').exists()
    by_dir = {}
    for file in folder.rglob('*.safetensors'):
        if is_diffusers and file.parent == folder:
            continue
        by_dir.setdefault(file.parent, []).append(file)
    chosen = []
    for files in by_dir.values():
        fp16 = [f for f in files if '.fp16.' in f.name]
        chosen.extend(fp16 if fp16 else files)
    return chosen

def safetensors_param_count(folder):
    """safetensors 헤더만 읽어 텐서 모양을 합산합니다. 가중치 전체를 메모리에 올리지 않습니다."""
    total = 0
    for file in weight_files(folder):
        with open(file, 'rb') as f:
            header_len = struct.unpack('<Q', f.read(8))[0]
            header = json.loads(f.read(header_len))
        for name, meta in header.items():
            if name == '__metadata__':
                continue
            n = 1
            for dim in meta['shape']:
                n *= dim
            total += n
    return total

def show_text_model(folder, cfg):
    print('  종류        : 텍스트 생성 (', cfg.get('model_type'), ')')
    print('  층 수       :', cfg.get('num_hidden_layers'))
    print('  은닉 크기   :', cfg.get('hidden_size'))
    print('  어텐션 헤드 :', cfg.get('num_attention_heads'), '/ KV 헤드', cfg.get('num_key_value_heads'))
    print('  어휘 수     :', cfg.get('vocab_size'))
    print('  최대 문맥   :', cfg.get('max_position_embeddings'), '토큰')
    gen = read_json(Path(folder) / 'generation_config.json')
    if gen:
        keys = ('do_sample', 'temperature', 'top_p', 'top_k', 'repetition_penalty', 'max_new_tokens')
        print('  생성 기본값 :', {k: gen[k] for k in keys if k in gen})

def show_diffusion_model(folder, index):
    print('  종류        : Diffusers 파이프라인 (', index.get('_class_name'), ')')
    parts = [k for k, v in index.items() if isinstance(v, list) and v[0] is not None]
    print('  구성 요소   :', ', '.join(parts))
    for part in ('unet', 'text_encoder', 'text_encoder_2', 'vae'):
        if (Path(folder) / part).is_dir():
            n = safetensors_param_count(Path(folder) / part)
            if n:
                print(f'    {part:15} {n/1e9:6.2f} B 파라미터')

for folder in MODEL_DIRS:
    print('\n' + '=' * 70)
    print(folder)
    if not Path(folder).is_dir():
        print('  폴더 없음 (사전 배포되지 않은 모델입니다)')
        continue
    cfg = read_json(Path(folder) / 'config.json')
    index = read_json(Path(folder) / 'model_index.json')
    if index:
        show_diffusion_model(folder, index)
    elif cfg:
        show_text_model(folder, cfg)
    params = safetensors_param_count(folder)
    size_gb = sum(f.stat().st_size for f in Path(folder).rglob('*') if f.is_file()) / 1024**3
    print(f'  전체 파라미터: {params/1e9:.2f} B   디스크 크기: {size_gb:.2f} GB')
    if params:
        est = {k: round(params * v / 1024**3, 1) for k, v in BYTES_PER_PARAM.items()}
        print('  가중치만의 예상 VRAM(GB):', est, ' + 작업 메모리 약 20~30%')

print('\n파라미터 수 × 바이트/파라미터 = 가중치 메모리. 문맥(KV 캐시)·중간 계산분은 따로 더합니다.')


C:\Users\User\models\qwen2.5-1.5b-instruct
  폴더 없음 (사전 배포되지 않은 모델입니다)

C:\Users\User\models\sdxl-turbo
  폴더 없음 (사전 배포되지 않은 모델입니다)

C:\Users\User\models\musicgen-small
  폴더 없음 (사전 배포되지 않은 모델입니다)

C:\Users\User\models\stable-video-diffusion-img2vid
  폴더 없음 (사전 배포되지 않은 모델입니다)

파라미터 수 × 바이트/파라미터 = 가중치 메모리. 문맥(KV 캐시)·중간 계산분은 따로 더합니다.


## 2. Qwen 텍스트

한 번에 한 조건만 바꿉니다. 바꾼 값, 고정한 값, 선택 이유를 함께 적습니다.

**2-1 Qwen 준비**

In [9]:
import copy
import time
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, set_seed

def prepare_model(repo_id, folder, marker, patterns):
    """서버 배포본을 먼저 쓰고, 없으면 내 홈 폴더에 한 번만 내려받습니다."""
    from pathlib import Path
    from huggingface_hub import snapshot_download
    shared = Path('/opt/models') / folder
    if (shared / marker).is_file():
        return str(shared)
    target = Path.home() / 'models' / folder
    done = target / '.download_complete'
    if done.is_file():
        return str(target)
    target.mkdir(parents=True, exist_ok=True)
    print('서버에 모델이 없어 내려받습니다. 몇 분 걸릴 수 있습니다:', target)
    try:
        snapshot_download(repo_id, local_dir=str(target), allow_patterns=patterns)
    except Exception as exc:
        raise RuntimeError(f'{repo_id} 모델을 내려받지 못했습니다. 인터넷 연결과 디스크 공간을 확인하고 스태프에게 알려 주세요.') from exc
    done.touch()
    return str(target)

if not torch.cuda.is_available():
    raise RuntimeError('CUDA GPU를 사용할 수 없습니다. 강사에게 문의하세요.')
if any(name in globals() for name in ('sdxl_pipe', 'img2img_pipe', 'qwen_model', 'music_model', 'video_pipe')):
    raise RuntimeError('기존 모델이 있습니다. 먼저 모델 해제 셀을 실행하세요.')

qwen_path = prepare_model('Qwen/Qwen2.5-1.5B-Instruct', 'qwen2.5-1.5b-instruct', 'config.json',
                          ['*.json', '*.txt', 'model.safetensors'])
qwen_tok = AutoTokenizer.from_pretrained(qwen_path, local_files_only=True)
qwen_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
qwen_model = AutoModelForCausalLM.from_pretrained(
    qwen_path, torch_dtype=qwen_dtype, local_files_only=True
).to('cuda').eval()
print('Qwen 준비 완료:', qwen_dtype)
print('GPU 점유 메모리(GB):', round(torch.cuda.memory_allocated() / 1024**3, 2), '← 슬라이드 18번 그림과 비교해 보세요')
print('모델 기본 생성 설정:', qwen_model.generation_config)

def qwen_generate(prompt, max_new_tokens=128, temperature=0.7, seed=42, greedy=False,
                  top_p=1.0, top_k=0, repetition_penalty=1.0):
    if 'qwen_model' not in globals() or 'qwen_tok' not in globals():
        raise RuntimeError('먼저 셀 2-1을 실행하세요.')
    messages = [{'role': 'user', 'content': prompt}]
    inputs = qwen_tok.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True, return_dict=True, return_tensors='pt'
    ).to('cuda')
    config = copy.deepcopy(qwen_model.generation_config)
    config.do_sample = not greedy
    config.temperature = 1.0 if greedy else temperature
    config.top_p, config.top_k = (1.0, 0) if greedy else (top_p, top_k)
    config.repetition_penalty, config.num_beams = repetition_penalty, 1
    if config.pad_token_id is None:
        eos = config.eos_token_id
        config.pad_token_id = eos[0] if isinstance(eos, (list, tuple)) else eos
    torch.cuda.synchronize()
    start = time.perf_counter()
    if not greedy:
        set_seed(seed)
    with torch.inference_mode():
        output = qwen_model.generate(**inputs, generation_config=config, max_new_tokens=max_new_tokens)
    torch.cuda.synchronize()
    elapsed = time.perf_counter() - start
    new_tokens = output[0, inputs['input_ids'].shape[-1]:]
    result = {'text': qwen_tok.decode(new_tokens, skip_special_tokens=True),
              'tokens': int(new_tokens.numel()), 'seconds': round(elapsed, 3),
              'settings': {'do_sample': config.do_sample, 'temperature': config.temperature,
                           'top_p': config.top_p, 'top_k': config.top_k,
                           'repetition_penalty': config.repetition_penalty, 'num_beams': config.num_beams,
                           'max_new_tokens': max_new_tokens, 'seed': None if greedy else seed}}
    print(result['text'], f"\n({result['tokens']} tokens, {result['seconds']}초)")
    return result

ModuleNotFoundError: No module named 'transformers'

**2-2 temperature 비교** `TEMPERATURES`나 `PROMPT`를 바꿔 다시 실행합니다.

In [ ]:
# ---- 바꿔 볼 값 ----
PROMPT = '대학생을 위한 과제 마감 알림 앱의 이름을 3개 제안해 줘. 각 이름의 뜻을 한 문장으로 설명해 줘.'
TEMPERATURES = (0.2, 1.2)
SEED = 42
MAX_NEW_TOKENS = 128
# --------------------
import json
from pathlib import Path

if 'qwen_generate' not in globals() or 'qwen_model' not in globals():
    raise RuntimeError('먼저 셀 2-1을 실행하세요.')
records = []
for temperature in TEMPERATURES:
    print(f'\n--- temperature={temperature}, seed={SEED} ---')
    result = qwen_generate(PROMPT, temperature=temperature, seed=SEED, max_new_tokens=MAX_NEW_TOKENS)
    records.append({'prompt': PROMPT, 'temperature': temperature, 'seed': SEED, 'max_new_tokens': MAX_NEW_TOKENS, 'greedy': False, **result})
print('\n--- greedy ---')
result = qwen_generate(PROMPT, max_new_tokens=MAX_NEW_TOKENS, greedy=True)
records.append({'prompt': PROMPT, 'temperature': None, 'seed': None, 'max_new_tokens': MAX_NEW_TOKENS, 'greedy': True, **result})
Path('results/qwen').mkdir(parents=True, exist_ok=True)
Path('results/qwen/temperature.json').write_text(json.dumps(records, ensure_ascii=False, indent=2), encoding='utf-8')

**2-3 설정 바꿔 가며 테스트** `SETTINGS` 목록에서 조건을 고치거나 한 줄을 추가합니다. 한 줄에 값 하나만 바꿉니다.

In [ ]:
# 아래 SETTINGS 목록을 직접 고쳐 가며 실행합니다. 한 줄에 한 조건만 바꾸는 것이 원칙입니다.
import json
from pathlib import Path

if 'qwen_generate' not in globals() or 'qwen_model' not in globals():
    raise RuntimeError('먼저 셀 2-1을 실행하세요.')

PROMPT = '대학생을 위한 과제 마감 알림 앱의 이름을 3개 제안해 줘. 각 이름의 뜻을 한 문장으로 설명해 줘.'

# 기준 조건 하나와, 기준에서 값 하나만 바꾼 조건들. 이름(label)은 자유롭게 적습니다.
BASE = dict(temperature=0.7, top_p=1.0, top_k=0, repetition_penalty=1.0, max_new_tokens=128, seed=42)
SETTINGS = [
    ('기준',               {}),
    ('temperature 0.3',    {'temperature': 0.3}),
    ('top_p 0.8',          {'top_p': 0.8}),
    ('repetition 1.2',     {'repetition_penalty': 1.2}),
    ('seed 7',             {'seed': 7}),
    # ('max_new_tokens 48', {'max_new_tokens': 48}),   # 주석을 풀어 추가합니다.
]

records = []
for label, change in SETTINGS:
    settings = {**BASE, **change}
    changed = ', '.join(f'{k}={v}' for k, v in change.items()) or '없음'
    print(f'\n=== {label} | 바꾼 값: {changed} ===')
    result = qwen_generate(PROMPT, **settings)
    records.append({'label': label, 'changed': change, 'settings': settings,
                    'text': result['text'], 'tokens': result['tokens'], 'seconds': result['seconds']})

print('\n--- 요약 ---')
print(f"{'조건':20} {'토큰':>5} {'초':>6}  첫 줄")
for r in records:
    first = r['text'].strip().splitlines()[0][:40] if r['text'].strip() else ''
    print(f"{r['label']:20} {r['tokens']:>5} {r['seconds']:>6}  {first}")
Path('results/qwen').mkdir(parents=True, exist_ok=True)
Path('results/qwen/settings_lab.json').write_text(json.dumps(records, ensure_ascii=False, indent=2), encoding='utf-8')
print('\n저장: results/qwen/settings_lab.json — 마음에 드는 조건과 이유를 기록하세요.')

**2-4 공지 정보 추출** `NOTICE`를 다른 공지로 바꿔 봅니다. 원문에 없는 내용을 만들어 내는지 대조합니다.

In [ ]:
# ---- 바꿔 볼 값 ----
NOTICE = '실습용 가상공지: 별빛 축제는 학생회관에서 오후 3시에 시작합니다. 참가비는 무료입니다.'
FIELDS = '행사명, 장소, 시작시간, 참가비, 담당자'
TEMPERATURE = 0.7
SEED = 42
MAX_NEW_TOKENS = 128
# --------------------
import json
from pathlib import Path

if 'qwen_generate' not in globals() or 'qwen_model' not in globals():
    raise RuntimeError('먼저 셀 2-1을 실행하세요.')
prompt = f'''다음 공지에서 {FIELDS}를 각각 한 줄로 추출하세요.
없는 정보는 정보 없음이라고 쓰세요.

{NOTICE}'''
records = []
for greedy in (True, False):
    print('\n---', 'greedy' if greedy else f'temperature={TEMPERATURE}, seed={SEED}', '---')
    result = qwen_generate(prompt, greedy=greedy, temperature=TEMPERATURE, seed=SEED, max_new_tokens=MAX_NEW_TOKENS)
    records.append({'prompt': prompt, 'greedy': greedy, 'temperature': None if greedy else TEMPERATURE,
                    'seed': None if greedy else SEED, 'max_new_tokens': MAX_NEW_TOKENS, **result})
print('\n생성 결과의 정확성은 보장되지 않습니다. 원문과 직접 비교하세요.')
Path('results/qwen').mkdir(parents=True, exist_ok=True)
Path('results/qwen/extract.json').write_text(json.dumps(records, ensure_ascii=False, indent=2), encoding='utf-8')

**선택 실습** 시간이 남을 때만 실행합니다. 출력 길이 비교입니다.

In [ ]:
# ---- 바꿔 볼 값 ----
PROMPT = '대학생을 위한 과제 마감 알림 앱의 이름을 3개 제안해 줘. 각 이름의 뜻을 한 문장으로 설명해 줘.'
LENGTHS = (32, 128)
# --------------------
import json
from pathlib import Path

if 'qwen_generate' not in globals() or 'qwen_model' not in globals():
    raise RuntimeError('먼저 셀 2-1을 실행하세요.')
records = []
for length in LENGTHS:
    print(f'\n--- max_new_tokens={length}, greedy ---')
    result = qwen_generate(PROMPT, max_new_tokens=length, greedy=True)
    records.append({'prompt': PROMPT, 'max_new_tokens': length, 'greedy': True, **result})
Path('results/qwen').mkdir(parents=True, exist_ok=True)
Path('results/qwen/length.json').write_text(json.dumps(records, ensure_ascii=False, indent=2), encoding='utf-8')

**선택 실습** top_p 비교입니다.

In [ ]:
# ---- 바꿔 볼 값 ----
PROMPT = '대학생을 위한 과제 마감 알림 앱의 이름을 3개 제안해 줘. 각 이름의 뜻을 한 문장으로 설명해 줘.'
TOP_P_VALUES = (0.5, 0.9, 1.0)
TEMPERATURE = 0.7
SEED = 42
# --------------------
import json
from pathlib import Path

if 'qwen_generate' not in globals() or 'qwen_model' not in globals():
    raise RuntimeError('먼저 셀 2-1을 실행하세요.')
records = []
for top_p in TOP_P_VALUES:
    print(f'\n--- top_p={top_p}, temperature={TEMPERATURE}, top_k=0, seed={SEED} ---')
    result = qwen_generate(PROMPT, temperature=TEMPERATURE, top_p=top_p, top_k=0, seed=SEED, max_new_tokens=128)
    records.append({'prompt': PROMPT, **result})
Path('results/qwen').mkdir(parents=True, exist_ok=True)
Path('results/qwen/top_p.json').write_text(json.dumps(records, ensure_ascii=False, indent=2), encoding='utf-8')

**2-5 모델 해제** 다음 모델로 넘어가기 전에 실행합니다. 저장한 결과 파일은 남습니다.

In [ ]:
import gc
import torch

# 이 실습에서 만든 모델과 헬퍼만 제거합니다. 다른 변수와 프로세스는 건드리지 않습니다.
for name in ('qwen_model', 'qwen_tok', 'qwen_generate', 'sdxl_pipe', 'img2img_pipe', 'sdxl_generate',
             'music_model', 'music_proc', 'music_generate', 'video_pipe', 'video_generate',
             'last_music_waveform', 'comparison_waveform', 'last_video_frames', 'video_source_image'):
    globals().pop(name, None)
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print('현재 PyTorch 프로세스가 점유한 GPU 메모리(MiB):',
          round(torch.cuda.memory_allocated() / 1024**2, 1))
print('실습 모델 참조를 해제했습니다. 다른 참조가 남아 있으면 메모리가 유지될 수 있습니다.')

## 3. SDXL-Turbo 이미지

**3-1 SDXL 준비**

In [ ]:
import json
import time
import uuid
from pathlib import Path
import torch
from diffusers import AutoPipelineForText2Image
from IPython.display import display

def prepare_model(repo_id, folder, marker, patterns):
    """서버 배포본을 먼저 쓰고, 없으면 내 홈 폴더에 한 번만 내려받습니다."""
    from pathlib import Path
    from huggingface_hub import snapshot_download
    shared = Path('/opt/models') / folder
    if (shared / marker).is_file():
        return str(shared)
    target = Path.home() / 'models' / folder
    done = target / '.download_complete'
    if done.is_file():
        return str(target)
    target.mkdir(parents=True, exist_ok=True)
    print('서버에 모델이 없어 내려받습니다. 몇 분 걸릴 수 있습니다:', target)
    try:
        snapshot_download(repo_id, local_dir=str(target), allow_patterns=patterns)
    except Exception as exc:
        raise RuntimeError(f'{repo_id} 모델을 내려받지 못했습니다. 인터넷 연결과 디스크 공간을 확인하고 스태프에게 알려 주세요.') from exc
    done.touch()
    return str(target)

if not torch.cuda.is_available():
    raise RuntimeError('CUDA GPU를 사용할 수 없습니다. 강사에게 문의하세요.')
if any(name in globals() for name in ('qwen_model', 'sdxl_pipe', 'img2img_pipe', 'music_model', 'video_pipe')):
    raise RuntimeError('기존 모델이 있습니다. 먼저 모델 해제 셀을 실행하세요.')

sdxl_path = prepare_model('stabilityai/sdxl-turbo', 'sdxl-turbo', 'model_index.json',
                          ['*.json', '*.txt', '*.fp16.safetensors'])
sdxl_pipe = AutoPipelineForText2Image.from_pretrained(
    sdxl_path, torch_dtype=torch.float16, variant='fp16', local_files_only=True
).to('cuda')
print('SDXL Turbo 준비 완료')
print('GPU 점유 메모리(GB):', round(torch.cuda.memory_allocated() / 1024**3, 2), '← 슬라이드 18번 그림과 비교해 보세요')

def sdxl_generate(prompt, seed=42, steps=1):
    if 'sdxl_pipe' not in globals():
        raise RuntimeError('먼저 셀 3-1을 실행하세요.')
    generator = torch.Generator(device='cpu').manual_seed(seed)
    torch.cuda.synchronize()
    start = time.perf_counter()
    with torch.inference_mode():
        image = sdxl_pipe(prompt=prompt, generator=generator, num_inference_steps=steps,
                          height=512, width=512, guidance_scale=0.0,
                          num_images_per_prompt=1).images[0]
    torch.cuda.synchronize()
    seconds = round(time.perf_counter() - start, 3)
    out = Path('results/sdxl')
    out.mkdir(parents=True, exist_ok=True)
    stem = 'text2img_' + uuid.uuid4().hex[:12]
    image.save(out / (stem + '.png'))
    metadata = {'prompt': prompt, 'seed': seed, 'steps': steps, 'width': 512, 'height': 512,
                'guidance_scale': 0.0, 'seconds': seconds, 'image': stem + '.png'}
    (out / (stem + '.json')).write_text(json.dumps(metadata, ensure_ascii=False, indent=2), encoding='utf-8')
    print(f'{seconds}초 | {out / (stem + ".png")}')
    display(image)
    return image, metadata

**3-2 seed 비교** `SEEDS`나 `PROMPT`를 바꿔 다시 실행합니다.

In [ ]:
# ---- 바꿔 볼 값 ----
PROMPT = 'cat in space suit photo'
SEEDS = (42, 100)
STEPS = 1
# --------------------
if 'sdxl_generate' not in globals() or 'sdxl_pipe' not in globals():
    raise RuntimeError('먼저 셀 3-1을 실행하세요.')
for seed in SEEDS:
    print(f'\n--- seed={seed}, steps={STEPS} ---')
    sdxl_generate(PROMPT, seed=seed, steps=STEPS)

**3-3 프롬프트 비교** `PROMPTS`에 내 서비스의 대표 이미지 설명을 넣어 봅니다.

In [ ]:
# ---- 바꿔 볼 값 ----
PROMPTS = [
    'cat in space suit photo',
    'cat in space suit photo, white background',
    'cat in space suit watercolor, white background',
]
SEED = 42
STEPS = 1
# --------------------
if 'sdxl_generate' not in globals() or 'sdxl_pipe' not in globals():
    raise RuntimeError('먼저 셀 3-1을 실행하세요.')
for prompt in PROMPTS:
    print(f'\n--- {prompt} | seed={SEED}, steps={STEPS} ---')
    sdxl_generate(prompt, seed=SEED, steps=STEPS)

**선택 실습** 생성 단계 비교입니다.

In [ ]:
# ---- 바꿔 볼 값 ----
PROMPT = 'cat in space suit photo'
STEPS_LIST = (1, 2, 4)
SEED = 42
# --------------------
if 'sdxl_generate' not in globals() or 'sdxl_pipe' not in globals():
    raise RuntimeError('먼저 셀 3-1을 실행하세요.')
for steps in STEPS_LIST:
    print(f'\n--- seed={SEED}, steps={steps} ---')
    sdxl_generate(PROMPT, seed=SEED, steps=steps)

**선택 실습** 이미지 변형(img2img)입니다.

In [ ]:
import json
import time
import uuid
from pathlib import Path
import torch
from diffusers import AutoPipelineForImage2Image
from IPython.display import display

if 'sdxl_pipe' not in globals() or 'sdxl_generate' not in globals():
    raise RuntimeError('먼저 셀 3-1을 실행하세요.')
if not hasattr(AutoPipelineForImage2Image, 'from_pipe'):
    raise RuntimeError('설치된 Diffusers에서 from_pipe를 지원하지 않습니다. 이 선택 실습을 건너뛰세요.')
if 'img2img_pipe' not in globals():
    try:
        img2img_pipe = AutoPipelineForImage2Image.from_pipe(sdxl_pipe)
    except (ValueError, TypeError, NotImplementedError) as exc:
        raise RuntimeError('현재 파이프라인에서는 img2img 변환을 지원하지 않습니다. 이 선택 실습을 건너뛰세요.') from exc

# 세 실험은 모두 같은 원본에서 시작하며 결과를 다음 입력으로 쓰지 않습니다.
source_image, source_info = sdxl_generate('cat in space suit photo', seed=42, steps=1)
prompt = 'cat in space suit watercolor, white background'
out = Path('results/sdxl')
for strength in (0.25, 0.5, 0.75):
    generator = torch.Generator(device='cpu').manual_seed(42)
    torch.cuda.synchronize()
    start = time.perf_counter()
    with torch.inference_mode():
        image = img2img_pipe(prompt=prompt, image=source_image, strength=strength,
                             generator=generator, num_inference_steps=4, guidance_scale=0.0,
                             num_images_per_prompt=1).images[0]
    torch.cuda.synchronize()
    seconds = round(time.perf_counter() - start, 3)
    stem = 'img2img_' + uuid.uuid4().hex[:12]
    image.save(out / (stem + '.png'))
    # 유효 단계 수는 이 실습의 고정 설정에서 계산한 값입니다.
    metadata = {'prompt': prompt, 'seed': 42, 'num_inference_steps': 4,
                'effective_steps': int(4 * strength), 'strength': strength,
                'guidance_scale': 0.0, 'width': 512, 'height': 512, 'seconds': seconds, 'source_image': source_info['image'],
                'image': stem + '.png'}
    (out / (stem + '.json')).write_text(json.dumps(metadata, ensure_ascii=False, indent=2), encoding='utf-8')
    print(f'strength={strength} | {seconds}초 | {out / (stem + ".png")}')
    display(image)

**영상에 쓸 이미지 메모:** results/sdxl/______________.png

**3-4 모델 해제** 다음 모델로 넘어가기 전에 실행합니다. 저장한 결과 파일은 남습니다.

In [ ]:
import gc
import torch

# 이 실습에서 만든 모델과 헬퍼만 제거합니다. 다른 변수와 프로세스는 건드리지 않습니다.
for name in ('qwen_model', 'qwen_tok', 'qwen_generate', 'sdxl_pipe', 'img2img_pipe', 'sdxl_generate',
             'music_model', 'music_proc', 'music_generate', 'video_pipe', 'video_generate',
             'last_music_waveform', 'comparison_waveform', 'last_video_frames', 'video_source_image'):
    globals().pop(name, None)
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print('현재 PyTorch 프로세스가 점유한 GPU 메모리(MiB):',
          round(torch.cuda.memory_allocated() / 1024**2, 1))
print('실습 모델 참조를 해제했습니다. 다른 참조가 남아 있으면 메모리가 유지될 수 있습니다.')

## 4. 선택 블록: 음악 또는 영상 중 하나

팀이 고른 쪽의 셀만 실행합니다.

### 4A. 음악 (MusicGen Small)

이어폰을 끼고 음량을 작게 맞춥니다.

**4A-1 MusicGen 준비** 아래 두 셀을 차례로 실행합니다. `MAX_MUSIC_SECONDS`는 한 번에 만들 최대 길이(초)이고, 토큰 수는 자동으로 계산됩니다.

In [ ]:
# ---- 바꿔 볼 값 ----
MAX_MUSIC_SECONDS = 10  # 한 번에 만들 최대 길이(초). 공유 실습은 10, 강사 시연은 30까지 올릴 수 있습니다.
# --------------------

In [ ]:
import copy
import json
import math
import time
import uuid
import wave
from pathlib import Path
import numpy as np
import torch
from IPython.display import Audio, display
from transformers import AutoProcessor, MusicgenForConditionalGeneration, set_seed

def prepare_model(repo_id, folder, marker, patterns):
    """서버 배포본을 먼저 쓰고, 없으면 내 홈 폴더에 한 번만 내려받습니다."""
    from pathlib import Path
    from huggingface_hub import snapshot_download
    shared = Path('/opt/models') / folder
    if (shared / marker).is_file():
        return str(shared)
    target = Path.home() / 'models' / folder
    done = target / '.download_complete'
    if done.is_file():
        return str(target)
    target.mkdir(parents=True, exist_ok=True)
    print('서버에 모델이 없어 내려받습니다. 몇 분 걸릴 수 있습니다:', target)
    try:
        snapshot_download(repo_id, local_dir=str(target), allow_patterns=patterns)
    except Exception as exc:
        raise RuntimeError(f'{repo_id} 모델을 내려받지 못했습니다. 인터넷 연결과 디스크 공간을 확인하고 스태프에게 알려 주세요.') from exc
    done.touch()
    return str(target)

if not torch.cuda.is_available():
    raise RuntimeError('CUDA GPU를 사용할 수 없습니다. 강사에게 문의하세요.')
if any(name in globals() for name in ('qwen_model', 'sdxl_pipe', 'img2img_pipe', 'music_model', 'video_pipe')):
    raise RuntimeError('기존 모델이 있습니다. 먼저 모델 해제 셀을 실행하세요.')

music_path = prepare_model('facebook/musicgen-small', 'musicgen-small', 'config.json',
                           ['*.json', '*.txt', '*.model', 'model.safetensors'])
MUSIC_DTYPE = torch.float16  # 무음·잡음·'유한하지 않은 값' 오류가 나면 torch.float32로 바꿔 다시 실행합니다.
music_proc = AutoProcessor.from_pretrained(music_path, local_files_only=True)
music_model = MusicgenForConditionalGeneration.from_pretrained(
    music_path, torch_dtype=MUSIC_DTYPE, local_files_only=True
).to('cuda').eval()
print('MusicGen 준비 완료:', MUSIC_DTYPE)
print('GPU 점유 메모리(GB):', round(torch.cuda.memory_allocated() / 1024**3, 2), '← 슬라이드 18번 그림과 비교해 보세요')

TOKENS_PER_SECOND = 50  # MusicGen은 1초에 약 50개의 소리 토큰을 만듭니다.
MODEL_MAX_SECONDS = 30  # 모델이 한 번에 만들 수 있는 최대 길이입니다.

def seconds_to_tokens(seconds):
    return int(round(seconds * TOKENS_PER_SECOND))

def music_limit_seconds():
    return min(MAX_MUSIC_SECONDS, MODEL_MAX_SECONDS)

def _music_sample(inputs, max_new_tokens, temperature, seed, guidance_scale, top_k, top_p):
    config = copy.deepcopy(music_model.generation_config)
    config.do_sample = True
    config.temperature, config.top_k, config.top_p = temperature, top_k, top_p
    config.guidance_scale, config.num_beams = guidance_scale, 1
    config.max_new_tokens = max_new_tokens
    torch.cuda.synchronize()
    start = time.perf_counter()
    set_seed(seed)
    with torch.inference_mode():
        audio = music_model.generate(**inputs, generation_config=config)
    torch.cuda.synchronize()
    elapsed = round(time.perf_counter() - start, 3)
    waveform = audio[0].float().cpu().numpy()  # (채널, 샘플)
    if waveform.ndim != 2 or waveform.shape[0] not in (1, 2):
        raise ValueError(f'지원하지 않는 음성 채널 모양입니다: {waveform.shape}')
    if waveform.size == 0 or not np.isfinite(waveform).all():
        raise ValueError('생성된 파형이 비어 있거나 유한하지 않은 값을 포함합니다.')
    return waveform, elapsed

def _music_save(waveform, metadata):
    sample_rate = metadata['sampling_rate']
    peak = float(np.max(np.abs(waveform)))
    waveform = np.clip(waveform, -1.0, 1.0)  # 조건별 음량 정규화는 적용하지 않습니다.
    out = Path('results/music')
    out.mkdir(parents=True, exist_ok=True)
    stem = 'music_' + uuid.uuid4().hex[:12]
    wav_path = out / (stem + '.wav')
    pcm = (waveform.T * 32767).astype('<i2')
    with wave.open(str(wav_path), 'wb') as wav:
        wav.setnchannels(waveform.shape[0])
        wav.setsampwidth(2)
        wav.setframerate(sample_rate)
        wav.writeframes(pcm.tobytes())
    metadata = {**metadata, 'channels': waveform.shape[0], 'duration': round(waveform.shape[1] / sample_rate, 3),
                'original_peak': peak, 'clipped': peak > 1.0, 'wav': str(wav_path)}
    (out / (stem + '.json')).write_text(json.dumps(metadata, ensure_ascii=False, indent=2), encoding='utf-8')
    print(f'실제 길이 {metadata["duration"]}초 | 생성 시간 {metadata["seconds"]}초 | {wav_path}')
    display(Audio(waveform, rate=sample_rate, normalize=False, autoplay=False))
    return waveform, metadata

def _music_crossfade(first, second, fade):
    """앞 음악의 끝과 뒤 음악의 시작을 fade 샘플만큼 겹칩니다."""
    ramp = np.linspace(0, np.pi / 2, fade, dtype=np.float32)
    mixed = first[:, -fade:] * np.cos(ramp) + second[:, :fade] * np.sin(ramp)
    return np.concatenate([first[:, :-fade], mixed, second[:, fade:]], axis=1)

def music_generate(prompt, max_new_tokens=256, temperature=1.0, seed=42,
                   guidance_scale=3.0, top_k=250, top_p=1.0, duration=None):
    """음악을 한 번에 만듭니다. duration(초)을 주면 max_new_tokens를 자동으로 계산합니다."""
    if 'music_model' not in globals() or 'music_proc' not in globals():
        raise RuntimeError('먼저 셀 4A-1을 실행하세요.')
    if not isinstance(prompt, str) or not prompt.strip():
        raise ValueError('음악을 설명할 텍스트를 입력하세요.')
    if duration is not None:
        max_new_tokens = seconds_to_tokens(duration)
    limit = seconds_to_tokens(music_limit_seconds())
    if not isinstance(max_new_tokens, int) or isinstance(max_new_tokens, bool) or not 32 <= max_new_tokens <= limit:
        raise ValueError(f'생성 길이는 {music_limit_seconds()}초({limit}토큰)까지입니다. 더 길게 만들려면 MAX_MUSIC_SECONDS를 올리세요. 최대 {MODEL_MAX_SECONDS}초입니다.')
    if not math.isfinite(temperature) or temperature <= 0 or not math.isfinite(guidance_scale) or guidance_scale <= 0:
        raise ValueError('temperature와 guidance_scale은 0보다 큰 유한한 숫자여야 합니다.')
    if not isinstance(top_k, int) or top_k < 0 or not math.isfinite(top_p) or not 0 < top_p <= 1:
        raise ValueError('top_k는 0 이상의 정수, top_p는 0보다 크고 1 이하인 숫자여야 합니다.')
    inputs = music_proc(text=[prompt], padding=True, return_tensors='pt').to('cuda')
    waveform, elapsed = _music_sample(inputs, max_new_tokens, temperature, seed, guidance_scale, top_k, top_p)
    return _music_save(waveform, {
        'prompt': prompt, 'max_new_tokens': max_new_tokens, 'temperature': temperature, 'seed': seed,
        'guidance_scale': guidance_scale, 'top_k': top_k, 'top_p': top_p, 'do_sample': True,
        'sampling_rate': int(music_model.config.audio_encoder.sampling_rate), 'seconds': elapsed})

def music_loop(waveform, metadata, total_seconds=30, crossfade_seconds=1.0):
    """같은 음악을 겹쳐 가며 반복합니다. GPU를 쓰지 않아 가장 빠릅니다."""
    sample_rate = metadata['sampling_rate']
    fade = int(crossfade_seconds * sample_rate)
    if fade < 1 or waveform.shape[1] <= 2 * fade:
        raise ValueError('음악이 겹칠 길이보다 충분히 길어야 합니다. crossfade_seconds를 줄이세요.')
    target = int(total_seconds * sample_rate)
    result = waveform
    while result.shape[1] < target:
        result = _music_crossfade(result, waveform, fade)
    return _music_save(result[:, :target], {**metadata, 'mode': 'loop', 'seconds': 0, 'source_wav': metadata.get('wav')})

def music_continue(waveform, metadata, total_seconds=30, overlap_seconds=5):
    """앞 음악의 끝부분을 입력으로 주고 뒷부분을 이어 만듭니다. 30초보다 길게 만들 수 있습니다."""
    if 'music_model' not in globals() or 'music_proc' not in globals():
        raise RuntimeError('먼저 셀 4A-1을 실행하세요.')
    sample_rate = metadata['sampling_rate']
    tail_size = int(overlap_seconds * sample_rate)
    fade = int(0.05 * sample_rate)
    if waveform.shape[1] <= tail_size:
        raise ValueError('음악이 overlap_seconds보다 길어야 합니다.')
    target = int(total_seconds * sample_rate)
    result, spent, seed = waveform, 0.0, metadata['seed']
    while result.shape[1] < target - int(0.2 * sample_rate):
        seed += 1
        remain = (target - result.shape[1]) / sample_rate
        chunk = max(1.0, min(music_limit_seconds() - overlap_seconds, remain))
        tail = result[:, -tail_size:].mean(axis=0)
        inputs = music_proc(audio=tail, sampling_rate=sample_rate, text=[metadata['prompt']],
                            padding=True, return_tensors='pt').to('cuda')
        inputs['input_values'] = inputs['input_values'].to(music_model.dtype)
        generated, elapsed = _music_sample(inputs, seconds_to_tokens(chunk), metadata['temperature'], seed,
                                           metadata['guidance_scale'], metadata['top_k'], metadata['top_p'])
        spent += elapsed
        print(f'이어 생성 중: {result.shape[1] / sample_rate:.1f}초 → 목표 {total_seconds}초')
        result = _music_crossfade(result, generated[:, tail_size - fade:], fade)
    return _music_save(result[:, :target], {**metadata, 'mode': 'continue', 'seconds': round(spent, 3),
                                            'overlap_seconds': overlap_seconds, 'source_wav': metadata.get('wav')})

**4A-2 생성 길이 비교** `MUSIC_PROMPT`나 `LENGTHS`를 바꿔 다시 실행합니다.

In [ ]:
# ---- 바꿔 볼 값 ----
MUSIC_PROMPT = 'Gentle instrumental background music with piano and acoustic guitar, calm and warm, no vocals.'
LENGTHS = (128, 256)
SEED = 42
# --------------------
if 'music_generate' not in globals() or 'music_model' not in globals():
    raise RuntimeError('먼저 셀 4A-1을 실행하세요.')
for length in LENGTHS:
    print(f'\n--- max_new_tokens={length}, seed={SEED} ---')
    last_music_waveform, last_music_metadata = music_generate(MUSIC_PROMPT, max_new_tokens=length, seed=SEED)

**4A-3 temperature 비교 (선택)** 4A-2를 기본값으로 실행한 뒤에만 동작합니다.

In [ ]:
from IPython.display import Audio, display

if 'music_generate' not in globals() or 'music_model' not in globals():
    raise RuntimeError('먼저 셀 4A-1을 실행하세요.')
if 'last_music_metadata' not in globals() or 'last_music_waveform' not in globals():
    raise RuntimeError('기준 음악이 없습니다. 먼저 셀 4A-2를 실행하세요.')
baseline = last_music_metadata
if any(baseline.get(k) != v for k, v in {'max_new_tokens': 256, 'temperature': 1.0,
       'seed': 42, 'guidance_scale': 3.0, 'top_k': 250, 'top_p': 1.0}.items()):
    raise RuntimeError('기준 음악의 설정이 다릅니다. 셀 4A-2를 기본값으로 다시 실행하세요.')
print('기준 음악: temperature=1.0 |', baseline['wav'])
display(Audio(last_music_waveform, rate=baseline['sampling_rate'], normalize=False, autoplay=False))
print('비교 음악: temperature=0.7; 나머지 설정은 동일합니다.')
comparison_waveform, comparison_metadata = music_generate(
    baseline['prompt'], max_new_tokens=256, temperature=0.7, seed=42,
    guidance_scale=3.0, top_k=250, top_p=1.0)

**선택 실습** 음악 길게 만들기입니다. 4A-2를 실행한 뒤 씁니다. 세 방법 중 쓸 줄의 `#`을 지웁니다.

In [ ]:
# ---- 바꿔 볼 값 ----
TARGET_SECONDS = 30
# --------------------
if 'music_generate' not in globals() or 'music_model' not in globals():
    raise RuntimeError('먼저 셀 4A-1을 실행하세요.')
if 'last_music_metadata' not in globals() or 'last_music_waveform' not in globals():
    raise RuntimeError('기준 음악이 없습니다. 먼저 셀 4A-2를 실행하세요.')

# 아래 셋 중 하나만 # 없이 둡니다. 다른 방법을 쓰려면 # 위치를 바꾸고 이 셀을 다시 실행합니다.

# 방법 1. 반복: 같은 음악을 겹쳐 이어 붙입니다. 가장 빠릅니다.
long_waveform, long_metadata = music_loop(last_music_waveform, last_music_metadata, total_seconds=TARGET_SECONDS)

# 방법 2. 이어 생성: 뒷부분을 새로 만듭니다. 시간이 더 걸리고 30초보다 길게도 됩니다.
# long_waveform, long_metadata = music_continue(last_music_waveform, last_music_metadata, total_seconds=TARGET_SECONDS)

# 방법 3. 한 번에 생성: 4A-1의 MAX_MUSIC_SECONDS를 30으로 올린 뒤 씁니다. 최대 30초입니다.
# long_waveform, long_metadata = music_generate(last_music_metadata['prompt'], duration=TARGET_SECONDS)

**4A-4 모델 해제** 다음 모델로 넘어가기 전에 실행합니다. 저장한 결과 파일은 남습니다.

In [ ]:
import gc
import torch

# 이 실습에서 만든 모델과 헬퍼만 제거합니다. 다른 변수와 프로세스는 건드리지 않습니다.
for name in ('qwen_model', 'qwen_tok', 'qwen_generate', 'sdxl_pipe', 'img2img_pipe', 'sdxl_generate',
             'music_model', 'music_proc', 'music_generate', 'video_pipe', 'video_generate',
             'last_music_waveform', 'comparison_waveform', 'last_video_frames', 'video_source_image'):
    globals().pop(name, None)
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print('현재 PyTorch 프로세스가 점유한 GPU 메모리(MiB):',
          round(torch.cuda.memory_allocated() / 1024**2, 1))
print('실습 모델 참조를 해제했습니다. 다른 참조가 남아 있으면 메모리가 유지될 수 있습니다.')

### 4B. 영상 (SVD)

**4B-1 입력 이미지 지정** 3번에서 고른 PNG 경로를 넣습니다. 비워 두면 가장 최근 이미지를 씁니다.

In [ ]:
VIDEO_IMAGE_PATH = ''  # 직접 입력하거나 비워 두면 results/sdxl의 최신 text2img PNG를 사용합니다.

**4B-2 SVD 준비**

In [ ]:
def ensure_packages(packages):
    """없는 패키지만 설치합니다. 이미 있는 torch 같은 패키지는 건드리지 않습니다."""
    import importlib
    import importlib.util
    import site
    import subprocess
    import sys
    missing = [name for module, name in packages.items() if importlib.util.find_spec(module) is None]
    if not missing:
        return
    print('필요한 패키지를 설치합니다:', ', '.join(missing))
    try:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', *missing], check=True, timeout=600)
    except (OSError, subprocess.SubprocessError) as exc:
        raise RuntimeError('패키지를 설치하지 못했습니다. 인터넷 연결을 확인하고 스태프에게 알려 주세요: ' + ', '.join(missing)) from exc
    user_site = site.getusersitepackages()
    if user_site not in sys.path:
        sys.path.append(user_site)
    importlib.invalidate_caches()
    if 'accelerate' in missing and 'diffusers' in sys.modules:
        raise RuntimeError('패키지를 설치했습니다. 위쪽 메뉴에서 Kernel > Restart Kernel을 실행한 뒤 셀 4B-1부터 다시 실행하세요. 저장한 이미지는 남아 있습니다.')

ensure_packages({'accelerate': 'accelerate', 'imageio': 'imageio', 'imageio_ffmpeg': 'imageio-ffmpeg'})

import json
import shutil
import subprocess
import time
import uuid
from pathlib import Path
import numpy as np
import torch
try:
    import imageio.v2 as imageio
    import imageio_ffmpeg
    import accelerate
except ImportError as exc:
    raise RuntimeError('영상 실습에 필요한 accelerate, imageio, imageio-ffmpeg를 운영자가 사전에 준비해야 합니다.') from exc
from PIL import Image, ImageOps
from IPython.display import Video, display
from diffusers import StableVideoDiffusionPipeline

def prepare_model(repo_id, folder, marker, patterns):
    """서버 배포본을 먼저 쓰고, 없으면 내 홈 폴더에 한 번만 내려받습니다."""
    from pathlib import Path
    from huggingface_hub import snapshot_download
    shared = Path('/opt/models') / folder
    if (shared / marker).is_file():
        return str(shared)
    target = Path.home() / 'models' / folder
    done = target / '.download_complete'
    if done.is_file():
        return str(target)
    target.mkdir(parents=True, exist_ok=True)
    print('서버에 모델이 없어 내려받습니다. 몇 분 걸릴 수 있습니다:', target)
    try:
        snapshot_download(repo_id, local_dir=str(target), allow_patterns=patterns)
    except Exception as exc:
        raise RuntimeError(f'{repo_id} 모델을 내려받지 못했습니다. 인터넷 연결과 디스크 공간을 확인하고 스태프에게 알려 주세요.') from exc
    done.touch()
    return str(target)

if not torch.cuda.is_available():
    raise RuntimeError('CUDA GPU를 사용할 수 없습니다. 강사에게 문의하세요.')
if any(name in globals() for name in ('qwen_model', 'sdxl_pipe', 'img2img_pipe', 'music_model', 'video_pipe')):
    raise RuntimeError('기존 모델이 있습니다. 먼저 모델 해제 셀을 실행하세요.')
try:
    ffmpeg_raw = imageio_ffmpeg.get_ffmpeg_exe()
    ffmpeg_resolved = shutil.which(ffmpeg_raw)
    if ffmpeg_resolved is None and Path(ffmpeg_raw).is_file():
        ffmpeg_resolved = str(Path(ffmpeg_raw).resolve())
    if ffmpeg_resolved is None:
        raise FileNotFoundError(ffmpeg_raw)
    subprocess.run([ffmpeg_resolved, '-version'], capture_output=True, text=True, check=True, timeout=10)
except (RuntimeError, OSError, subprocess.SubprocessError) as exc:
    raise RuntimeError('로컬 ffmpeg를 실행하지 못했습니다. 운영자에게 패키지와 바이너리 사전 준비를 요청하세요.') from exc
ffmpeg_path = ffmpeg_resolved
print('로컬 ffmpeg:', ffmpeg_path)

if VIDEO_IMAGE_PATH:
    video_source_path = Path(VIDEO_IMAGE_PATH).expanduser()
else:
    candidates = list(Path('results/sdxl').glob('text2img_*.png'))
    if not candidates:
        raise FileNotFoundError('results/sdxl에 이미지가 없습니다. SDXL 이미지를 먼저 만들거나 VIDEO_IMAGE_PATH를 지정하세요.')
    video_source_path = max(candidates, key=lambda path: path.stat().st_mtime)
if not video_source_path.is_file():
    raise FileNotFoundError(f'입력 이미지가 없습니다: {video_source_path}')
video_source_path = video_source_path.resolve()
with Image.open(video_source_path) as original:
    video_source_image = ImageOps.pad(original.convert('RGB'), (1024, 576), color='black')
video_out = Path('results/video')
video_out.mkdir(parents=True, exist_ok=True)
video_prepared_path = video_out / ('input_' + uuid.uuid4().hex[:12] + '.png')
video_source_image.save(video_prepared_path)
print('선택한 원본:', video_source_path, '| 비율을 유지한 변환 이미지:', video_prepared_path)
display(video_source_image)

video_path = prepare_model('stabilityai/stable-video-diffusion-img2vid', 'stable-video-diffusion-img2vid', 'model_index.json',
                           ['*.json', '*.txt', '*.fp16.safetensors'])
video_pipe = StableVideoDiffusionPipeline.from_pretrained(
    video_path, torch_dtype=torch.float16,
    variant='fp16', local_files_only=True
)
if getattr(video_pipe.unet.config, 'num_frames', None) != 14:
    del video_pipe
    raise RuntimeError('14프레임 모델이 아닙니다. XT가 아닌 stable-video-diffusion-img2vid 경로를 확인하세요.')
video_pipe.enable_model_cpu_offload()
video_pipe.unet.enable_forward_chunking()
print('영상 모델 준비 완료 (CPU offload: GPU 점유는 생성 중에 늘어납니다)')
print('GPU 점유 메모리(GB):', round(torch.cuda.memory_allocated() / 1024**3, 2), '← 슬라이드 18번 그림과 비교해 보세요')

def video_generate(motion_bucket_id=64, seed=42):
    if 'video_pipe' not in globals():
        raise RuntimeError('먼저 셀 4B-2을 실행하세요.')
    generator = torch.Generator(device='cpu').manual_seed(seed)
    torch.cuda.synchronize()
    start = time.perf_counter()
    with torch.inference_mode():
        # decode_chunk_size=1은 메모리를 줄이지만 프레임 깜박임이 생길 수 있습니다.
        frames = video_pipe(image=video_source_image, generator=generator, num_frames=14, height=576, width=1024,
                            num_inference_steps=25, fps=7, motion_bucket_id=motion_bucket_id,
                            min_guidance_scale=1.0, max_guidance_scale=3.0,
                            noise_aug_strength=0.02, decode_chunk_size=1).frames[0]
    torch.cuda.synchronize()
    seconds = round(time.perf_counter() - start, 3)
    stem = 'svd_' + uuid.uuid4().hex[:12]
    output_path = video_out / (stem + '.mp4')
    imageio.mimsave(str(output_path), [np.asarray(frame.convert('RGB')) for frame in frames], fps=7)
    metadata = {'source_image': str(video_source_path), 'prepared_image': str(video_prepared_path),
                'motion_bucket_id': motion_bucket_id, 'seed': seed, 'num_frames': len(frames),
                'num_inference_steps': 25, 'fps': 7, 'generation_fps': 7, 'export_fps': 7,
                'duration': len(frames) / 7,
                'min_guidance_scale': 1.0, 'max_guidance_scale': 3.0, 'noise_aug_strength': 0.02,
                'decode_chunk_size': 1, 'seconds': seconds, 'video': str(output_path)}
    (video_out / (stem + '.json')).write_text(json.dumps(metadata, ensure_ascii=False, indent=2), encoding='utf-8')
    print(f'{seconds}초 | {output_path}')
    display(Video(str(output_path), embed=True))
    return frames, metadata

**4B-3 클립 생성** 결과가 나올 때까지 다시 실행하지 않습니다.

In [ ]:
# ---- 바꿔 볼 값 ----
MOTION_BUCKET_ID = 64
SEED = 42
RUN_SECOND_CLIP = False  # 강사가 리허설 후 허용한 경우에만 True로 바꿉니다.
SECOND_MOTION_BUCKET_ID = 180
# --------------------
if 'video_generate' not in globals() or 'video_pipe' not in globals():
    raise RuntimeError('먼저 셀 4B-2을 실행하세요.')
print(f'motion_bucket_id={MOTION_BUCKET_ID}, seed={SEED}')
last_video_frames, last_video_metadata = video_generate(motion_bucket_id=MOTION_BUCKET_ID, seed=SEED)
if RUN_SECOND_CLIP:
    print(f'motion_bucket_id={SECOND_MOTION_BUCKET_ID}, seed={SEED}')
    last_video_frames, last_video_metadata = video_generate(motion_bucket_id=SECOND_MOTION_BUCKET_ID, seed=SEED)
else:
    print('대조 클립은 RUN_SECOND_CLIP=True로 바꾸어 실행할 수 있습니다. 강사 안내를 따르세요.')

**4B-4 재생 속도 비교 (선택)** 새 프레임을 만들지 않습니다.

In [ ]:
# ---- 바꿔 볼 값 ----
EXPORT_FPS_LIST = (4, 8)
# --------------------
import json
import uuid
from pathlib import Path
import numpy as np
import imageio.v2 as imageio
from IPython.display import Video, display

if 'last_video_frames' not in globals() or 'last_video_metadata' not in globals():
    raise RuntimeError('먼저 셀 4B-3을 실행하세요.')
print('동일한 프레임의 재생 속도를 비교합니다. 새 동작을 생성하지 않습니다.')
out = Path('results/video')
out.mkdir(parents=True, exist_ok=True)
for export_fps in EXPORT_FPS_LIST:
    stem = 'playback_' + uuid.uuid4().hex[:12]
    path = out / (stem + '.mp4')
    imageio.mimsave(str(path), [np.asarray(frame.convert('RGB')) for frame in last_video_frames], fps=export_fps)
    metadata = {'source_video': last_video_metadata['video'], 'generation_fps': last_video_metadata.get('generation_fps', 7),
                'export_fps': export_fps, 'num_frames': len(last_video_frames),
                'duration': len(last_video_frames) / export_fps, 'video': str(path)}
    (out / (stem + '.json')).write_text(json.dumps(metadata, ensure_ascii=False, indent=2), encoding='utf-8')
    print(f'{export_fps}fps | {metadata["duration"]}초 | {path}')
    display(Video(str(path), embed=True))

**4B-5 모델 해제** 다음 모델로 넘어가기 전에 실행합니다. 저장한 결과 파일은 남습니다.

In [ ]:
import gc
import torch

# 이 실습에서 만든 모델과 헬퍼만 제거합니다. 다른 변수와 프로세스는 건드리지 않습니다.
for name in ('qwen_model', 'qwen_tok', 'qwen_generate', 'sdxl_pipe', 'img2img_pipe', 'sdxl_generate',
             'music_model', 'music_proc', 'music_generate', 'video_pipe', 'video_generate',
             'last_music_waveform', 'comparison_waveform', 'last_video_frames', 'video_source_image'):
    globals().pop(name, None)
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print('현재 PyTorch 프로세스가 점유한 GPU 메모리(MiB):',
          round(torch.cuda.memory_allocated() / 1024**2, 1))
print('실습 모델 참조를 해제했습니다. 다른 참조가 남아 있으면 메모리가 유지될 수 있습니다.')

In [2]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA 버전:", torch.version.cuda)
print("CUDA 사용 가능:", torch.cuda.is_available())

!nvidia-smi


PyTorch: 2.15.0+cpu
CUDA 버전: None
CUDA 사용 가능: False


'nvidia-smi' is not recognized as an internal or external command,
operable program or batch file.


In [10]:
import os
import sys
import platform
import torch

print("현재 운영체제:", platform.system())
print("현재 Python:", sys.executable)
print("현재 작업 폴더:", os.getcwd())
print("홈 폴더:", os.path.expanduser("~"))
print("PyTorch:", torch.__version__)
print("CUDA:", torch.version.cuda)
print("GPU 사용 가능:", torch.cuda.is_available())

print("\n모델 폴더 확인")
for path in ["/opt/models", os.path.expanduser("~/models")]:
    print(path, "존재함" if os.path.isdir(path) else "없음")

현재 운영체제: Windows
현재 Python: C:\Users\User\AppData\Local\Programs\Python\Python314\python.exe
현재 작업 폴더: C:\대학\해커톤\city-eye
홈 폴더: C:\Users\User
PyTorch: 2.15.0+cpu
CUDA: None
GPU 사용 가능: False

모델 폴더 확인
/opt/models 없음
C:\Users\User/models 없음


## 5. 정리

- 텍스트, 이미지, 선택 블록 중 하나를 골라 비교 결과 두 개를 찾습니다.
- 바꾼 조건과 고정한 조건, 선택한 결과와 이유를 한 문장으로 적습니다.
- 허깅페이스에서 찾은 후보 모델 하나와 필요한 VRAM을 적습니다.

결과 파일: `results/qwen/*.json`, `results/sdxl/*.png`, `results/music/*.wav`, `results/video/*.mp4`

**공유 문장:** 저희는 __________에 쓸 결과를 만들었습니다. __________ 조건을 비교했고, __________ 때문에 이 결과를 선택했습니다.